In [1]:
# pip install pymatgen
import os
from pymatgen.ext.matproj import MPRester
from pymatgen.analysis.phase_diagram import PhaseDiagram

# 0) Make sure your API key is set:
# export MAPI_KEY="YOUR_KEY_HERE"
api_key = 'pJx41Nd1Bl2zMSeN8JPeo9a65TJG0dAY'
# os.environ.get("pJx41Nd1Bl2zMSeN8JPeo9a65TJG0dAY")

import requests

In [3]:
session = requests.Session()
session.verify = False  # ⚠️ insecure, only for debugging

elements = ["Li", "Fe", "P", "O"]   # <-- your chemistry
with MPRester(api_key) as mpr:
    # 1) Fetch entries with formation energies for the given chemical system
    entries = mpr.get_entries_in_chemsys(elements)

    # 2) Build phase diagram and get stable entries
    pd = PhaseDiagram(entries)
    stable_entries = pd.stable_entries

    # 3) Gather energies and structures
    results = []
    for e in stable_entries:
        # entry_id is typically the MP material_id for ComputedEntry
        mp_id = getattr(e, "entry_id", None)
        formula = e.composition.reduced_formula
        E_form_pa = pd.get_form_energy_per_atom(e)          # formation energy/atom
        E_above = pd.get_e_above_hull(e)                    # 0 for hull entries

        # 4) Fetch a representative structure from MP
        structure = None
        if mp_id:
            try:
                structure = mpr.get_structure_by_material_id(mp_id)  # pymatgen Structure
            except Exception:
                pass

        results.append({
            "mp_id": mp_id,
            "formula": formula,
            "E_form_per_atom_eV": E_form_pa,
            "E_above_hull_eV_per_atom": E_above,
            "structure": structure,
        })

MPRestError: HTTPSConnectionPool(host='api.materialsproject.org', port=443): Max retries exceeded with url: /materials/thermo/?_fields=entries&chemsys=Li,Fe,P,O,Fe-Li,Li-P,Li-O,Fe-P,Fe-O,O-P,Fe-Li-P,Fe-Li-O,Li-O-P,Fe-O-P,Fe-Li-O-P&_per_page=1000&_page=1 (Caused by SSLError(SSLError(136, '[X509: NO_CERTIFICATE_OR_CRL_FOUND] no certificate or crl found (_ssl.c:4149)')))

In [ ]:




    # 5) (Optional) write CIFs
    from pathlib import Path
    outdir = Path("mp_hull_cifs"); outdir.mkdir(exist_ok=True)
    for r in results:
        if r["structure"] is not None and r["mp_id"] is not None:
            r["structure"].to(fmt="cif", filename=str(outdir / f'{r["formula"]}_{r["mp_id"]}.cif'))
